In [10]:
# Instala de forma silenciosa la librería de Azure Cognitive Services Speech.
# La opción -q reduce la salida en consola durante la instalación.
!pip -q install azure-cognitiveservices-speech

In [11]:
# Importa el módulo os para trabajar con variables de entorno y rutas del sistema.
import os

# Importa el SDK de Azure Speech y lo renombra como speechsdk para usarlo más fácil.
import azure.cognitiveservices.speech as speechsdk

# Define la clave de acceso del servicio Speech de Azure como variable de entorno.
os.environ["SPEECH_KEY"] = "8wv9xzqLEOHzU0DZsa9yRK0r7ze2ZzFoTd9JiTbZlY5TFy4EFRenJQQJ99CCACMsfrFXJ3w3AAAYACOGxrxI"

# Define la región del servicio Speech de Azure como variable de entorno.
os.environ["SPEECH_REGION"] = "westus3"

In [12]:
# Usa ffmpeg para convertir el archivo de audio original (.m4a) a formato .wav,
# que suele ser más compatible para procesamiento de voz.
# -ac 1       -> convierte el audio a un solo canal (mono)
# -ar 16000   -> ajusta la frecuencia de muestreo a 16000 Hz
# -sample_fmt s16 -> usa formato de muestra de 16 bits
!ffmpeg -y -i "speechToText.m4a" -ac 1 -ar 16000 -sample_fmt s16 audio.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [13]:
# Muestra información básica del archivo generado para verificar su formato.
!file audio.wav

audio.wav: RIFF (little-endian) data, WAVE audio, Microsoft PCM, 16 bit, mono 16000 Hz


In [14]:

# Recupera la clave Speech desde las variables de entorno.
SPEECH_KEY = os.environ["SPEECH_KEY"]

# Recupera la región Speech desde las variables de entorno.
SPEECH_REGION = os.environ["SPEECH_REGION"]

# Define el nombre del archivo de audio que se usará para la transcripción.
# Aquí se asume que el archivo convertido se llama "audio.wav".
audio_file = "audio.wav"  # <-- cambia al nombre real que subiste

# Crea la configuración principal del servicio Speech usando la clave y la región.
speech_config = speechsdk.SpeechConfig(subscription=SPEECH_KEY, region=SPEECH_REGION)

# Define el idioma esperado del audio para mejorar el reconocimiento.
# En este caso se configura español de México.
speech_config.speech_recognition_language = "es-MX"

# Configura la fuente de audio a partir de un archivo local.
audio_config = speechsdk.audio.AudioConfig(filename=audio_file)

# Crea el reconocedor de voz usando la configuración del servicio y el archivo de audio.
recognizer = speechsdk.SpeechRecognizer(
    speech_config=speech_config,
    audio_config=audio_config)

# Muestra un mensaje en consola para indicar que la transcripción ha comenzado.
print("Transcribiendo...")

# Ejecuta una transcripción única del audio.
# recognize_once() funciona bien para fragmentos cortos;
# si el audio es largo, puede no transcribirlo completo.
result = recognizer.recognize_once()

# Si el reconocimiento fue exitoso y se detectó voz,
# imprime el texto transcrito.
if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print(" Texto reconocido:")
    print(result.text)

# Si no se pudo reconocer voz en el archivo,
# muestra un mensaje y detalles del problema.
elif result.reason == speechsdk.ResultReason.NoMatch:
    print("No se reconoció voz en el audio.")
    print(result.no_match_details)

# Si la transcripción fue cancelada por algún error o problema de configuración,
# obtiene y muestra información detallada del motivo.
elif result.reason == speechsdk.ResultReason.Canceled:
    cancellation = result.cancellation_details
    print(" Cancelado:", cancellation.reason)
    print("Detalles:", cancellation.error_details)

Transcribiendo...
 Texto reconocido:
Hola, soy José Ignacio. Esto es una prueba de azure speech to text en español de México. Estoy transcribiendo un audio corto para validar el servicio y documentar los resultados.
